# 22 — PubMedBERT NER + OLS Pipeline

**Architecture (fast, Kaggle-compatible):**
```
1. PubMedBERT NER   → entities from abstract  (~1s per PXD vs ~4min for Qwen)
2. OLS resolution   → canonical NT=;AC= format for every extracted value
3. Regex            → protocol fields (instrument, enzyme, mods, tolerances)
4. GPT correction   → maps GPT-extracted values to gold SDRF format
5. TF-IDF fallback  → borrows values from similar training PXDs
6. Checkpoint save  → crash-safe, resumes mid-run
```

**Key format rules (from competition spec + training SDRF inspection):**
- `Organism`: `9606 (Homo sapiens)` — taxon ID first
- `OrganismPart`: `NT=blood serum;AC=UBERON:0001977` — UBERON ontology
- `Disease`: plain text — `COVID-19`, `Alzheimer disease` — NO NT= wrapper
- `CellLine`: plain text — `HEK293T`, `HeLa` — NO NT= wrapper
- `Instrument`: `AC=MS:1003094;NT=Orbitrap Exploris 480` — AC= FIRST
- `Not Applicable` — capital N, capital A (competition standard)

## [SETUP] Install

In [1]:
import subprocess, sys
from pathlib import Path

_is_kaggle = Path('/kaggle').exists()

PKGS = ['transformers>=4.40','accelerate','scikit-learn','rapidfuzz','requests','tqdm']

if _is_kaggle:
    for pkg in PKGS:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
    print('Kaggle packages installed')
else:
    import importlib
    PKG_MAP = {'transformers':'transformers>=4.40','accelerate':'accelerate',
               'sklearn':'scikit-learn','rapidfuzz':'rapidfuzz'}
    missing = [pip for mod,pip in PKG_MAP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        for pkg in missing:
            subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
    print(f'Local: {"installed "+str(missing) if missing else "all packages present"}')

Local: all packages present


## [SETUP] Paths

In [2]:
import os, json, re, difflib, time, warnings, pickle
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

IS_KAGGLE = Path('/kaggle').exists()

if IS_KAGGLE:
    COMPETITION_DATA = Path('/kaggle/input/harmonizing-the-data-of-your-data')
    TRAIN_TEXT_DIR   = COMPETITION_DATA / 'Training_PubText' / 'PubText'
    TRAIN_SDRF_DIR   = COMPETITION_DATA / 'Training_SDRFs'  / 'HarmonizedFiles'
    TRAIN_GPT_DIR    = COMPETITION_DATA / 'Training_GPT_Extract' / 'GPT_Extract' / 'o4-mini-2025-04-16'
    TEST_TEXT_DIR    = COMPETITION_DATA / 'Test PubText' / 'Test PubText'
    BASELINE_PROMPT  = COMPETITION_DATA / 'BaselinePrompt.txt'
    SAMPLE_SUB       = COMPETITION_DATA / 'SampleSubmission.csv'
    SCORING_PY       = COMPETITION_DATA / 'Scoring.py'
    OUTPUT_DIR       = Path('/kaggle/working')
    # PubMedBERT NER model — HF Hub (requires internet ON in notebook settings)
    # Or upload model files as dataset 'pubmedbert-ner' and point to /kaggle/input/pubmedbert-ner
    NER_MODEL = 'd4data/biomedical-ner-all'
else:
    BASE_DIR = Path.cwd().parent.resolve()
    DATA_DIR = BASE_DIR / 'data'
    _tt = [DATA_DIR/'Training_PubText'/'PubText', DATA_DIR/'Training_PubText', DATA_DIR/'TrainingPubText']
    TRAIN_TEXT_DIR = next((p for p in _tt if p.exists()), _tt[0])
    _ts = [DATA_DIR/'Training_SDRFs'/'HarmonizedFiles', DATA_DIR/'Training_SDRFs', DATA_DIR/'TrainingSDRFs']
    TRAIN_SDRF_DIR = next((p for p in _ts if p.exists()), _ts[0])
    _tg = [DATA_DIR/'Training_GPT_Extract'/'GPT_Extract'/'o4-mini-2025-04-16', DATA_DIR/'Training_GPT_Extract']
    TRAIN_GPT_DIR  = next((p for p in _tg if p.exists()), _tg[0])
    _te = [DATA_DIR/'Test_PubText'/'Test PubText', DATA_DIR/'Test PubText'/'Test PubText',
           DATA_DIR/'Test_PubText', DATA_DIR/'TestPubText']
    TEST_TEXT_DIR  = next((p for p in _te if p.exists()), _te[0])
    BASELINE_PROMPT = DATA_DIR / 'BaselinePrompt.txt'
    SAMPLE_SUB      = DATA_DIR / 'SampleSubmission.csv'
    SCORING_PY      = BASE_DIR / 'src' / 'Scoring.py'
    OUTPUT_DIR      = BASE_DIR / 'outputs'
    NER_MODEL       = 'd4data/biomedical-ner-all'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MAX_PAPER_CHARS = 4_000
NOT_APPLICABLE  = 'Not Applicable'

print(f'IS_KAGGLE      : {IS_KAGGLE}')
for name,path in [('TRAIN_TEXT',TRAIN_TEXT_DIR),('TRAIN_SDRF',TRAIN_SDRF_DIR),
                   ('TRAIN_GPT',TRAIN_GPT_DIR),('TEST_TEXT',TEST_TEXT_DIR),
                   ('SAMPLE_SUB',SAMPLE_SUB)]:
    print(f'  {name:<12}: {path}  exists={path.exists()}')

IS_KAGGLE      : False
  TRAIN_TEXT  : C:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\data\TrainingPubText  exists=True
  TRAIN_SDRF  : C:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\data\TrainingSDRFs  exists=True
  TRAIN_GPT   : C:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\data\Training_GPT_Extract\GPT_Extract\o4-mini-2025-04-16  exists=True
  TEST_TEXT   : C:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\data\TestPubText  exists=True
  SAMPLE_SUB  : C:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\data\SampleSubmission.csv  exists=True


c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## [SETUP] Load data

In [3]:
def load_json_text(path):
    with open(path, encoding='utf-8', errors='replace') as f:
        data = json.load(f)
    if 'full_text' not in data:
        parts = []
        for k in ('TITLE','ABSTRACT','METHODS','MATERIALS AND METHODS',
                  'title','abstract','body','text','content'):
            v = data.get(k,'')
            if isinstance(v,list): v = ' '.join(str(x) for x in v)
            if v.strip(): parts.append(v.strip())
        data['full_text'] = ' '.join(parts) or ' '.join(
            str(v) for v in data.values() if isinstance(v,str))[:8000]
    return data

def load_gpt_txt(path):
    result = {}
    with open(path, encoding='utf-8', errors='replace') as f:
        for line in f:
            line = line.strip()
            if not line or ':' not in line: continue
            k,_,v = line.partition(':'); k=k.strip(); v=v.strip()
            if not k: continue
            result[k] = [result[k],v] if k in result and not isinstance(result[k],list) else \
                        result[k]+[v] if isinstance(result.get(k),list) else v
    return result

template_df = pd.read_csv(SAMPLE_SUB)
ALL_COLS  = list(template_df.columns)
META_COLS = [c for c in ALL_COLS if c not in ('ID','PXD','Raw Data File','Usage')]
print(f'Template: {len(ALL_COLS)} cols, {len(META_COLS)} metadata')

train_papers: Dict[str,dict] = {}
for fp in sorted(TRAIN_TEXT_DIR.glob('*.json')):
    train_papers[fp.stem.split('_')[0]] = load_json_text(fp)
print(f'Train papers : {len(train_papers)}')

train_sdrfs: Dict[str,pd.DataFrame] = {}
for fp in list(TRAIN_SDRF_DIR.glob('*.tsv')) + list(TRAIN_SDRF_DIR.glob('*.csv')):
    pxd = fp.stem.split('_')[0]
    try: train_sdrfs[pxd] = pd.read_csv(fp, sep='\t' if fp.suffix=='.tsv' else ',', low_memory=False)
    except: pass
print(f'Train SDRFs  : {len(train_sdrfs)}')

train_gpt: Dict[str,dict] = {}
if TRAIN_GPT_DIR.exists():
    for fp in sorted(TRAIN_GPT_DIR.glob('*.txt')):
        train_gpt[fp.stem.split('_')[0]] = load_gpt_txt(fp)
    for fp in sorted(TRAIN_GPT_DIR.glob('*.json')):
        pxd = fp.stem.split('_')[0]
        if pxd not in train_gpt:
            with open(fp) as f: train_gpt[pxd] = json.load(f)
print(f'GPT extracts : {len(train_gpt)}')

test_papers: Dict[str,dict] = {}
for fp in sorted(TEST_TEXT_DIR.glob('*.json')):
    test_papers[fp.stem.split('_')[0]] = load_json_text(fp)
print(f'Test papers  : {len(test_papers)}')
print(f'Test PXDs    : {sorted(test_papers.keys())}')

Template: 81 cols, 77 metadata
Train papers : 104
Train SDRFs  : 103
GPT extracts : 103
Test papers  : 16
Test PXDs    : ['PXD004010', 'PXD016436', 'PXD019519', 'PXD025663', 'PXD040582', 'PXD050621', 'PXD061009', 'PXD061090', 'PXD061136', 'PXD061195', 'PXD061285', 'PXD062014', 'PXD062469', 'PXD062877', 'PXD064564', 'PubText']


## [FUNC] OLS resolution

In [5]:
import requests
from functools import lru_cache

# ── OLS disk cache — load pre-fetched terms to avoid API calls ───────────────
_ols_cache_path = OUTPUT_DIR / "ols_cache.pkl" if OUTPUT_DIR.exists() else None
_ols_cache: Dict[str, Optional[str]] = {}
if _ols_cache_path and _ols_cache_path.exists():
    import pickle
    with open(_ols_cache_path, "rb") as _f:
        _ols_cache = pickle.load(_f)
    print(f"OLS disk cache loaded: {len(_ols_cache)} entries")

OLS_BASE    = "https://www.ebi.ac.uk/ols4/api/search"
OLS_TIMEOUT = 8   # seconds per request — fail fast rather than hang

# Column → (ontology, format_string)
# format_string uses {label} and {obo_id} placeholders
COLUMN_ONTOLOGY: Dict[str, Tuple[str, str]] = {
    "Characteristics[Organism]":         ("ncbitaxon", "{obo_id} ({label})"),  # taxon ID format
    "Characteristics[Disease]":          (None,        "{label}"),          # plain text — no ontology wrapping
    "Characteristics[OrganismPart]":     ("uberon",    "NT={label};AC={obo_id}"),
    "Characteristics[CellType]":         ("cl",        "NT={label};AC={obo_id}"),
    "Characteristics[CellLine]":         ("efo",       "NT={label};AC={obo_id}"),
    "Characteristics[DevelopmentalStage]":("efo",      "NT={label};AC={obo_id}"),
    "Characteristics[Sex]":              ("pato",      "NT={label};AC={obo_id}"),
    "Characteristics[MaterialType]":     ("efo",       "NT={label};AC={obo_id}"),
    "Characteristics[CleavageAgent]":    ("ms",        "NT={label};AC={obo_id}"),
    "Characteristics[AlkylationReagent]":("ms",        "NT={label};AC={obo_id}"),
    "Characteristics[ReductionReagent]": ("ms",        "NT={label};AC={obo_id}"),
    "Characteristics[Label]":            ("ms",        "NT={label};AC={obo_id}"),
    "Characteristics[Modification]":     ("ms",        "NT={label};MT=Variable;AC={obo_id}"),
    "Characteristics[Modification].1":   ("ms",        "NT={label};MT=Variable;AC={obo_id}"),
    "Characteristics[Modification].2":   ("ms",        "NT={label};MT=Variable;AC={obo_id}"),
    "Characteristics[Modification].3":   ("ms",        "NT={label};MT=Fixed;AC={obo_id}"),
    "Characteristics[Modification].4":   ("ms",        "NT={label};MT=Fixed;AC={obo_id}"),
    "Comment[Instrument]":               ("ms",        "AC={obo_id};NT={label}"),  # AC= first for instruments
    "Comment[FragmentationMethod]":      ("ms",        "AC={obo_id};NT={label}"),
    "Comment[MS2MassAnalyzer]":          ("ms",        "AC={obo_id};NT={label}"),  # fixed
    "Comment[IonizationType]":           ("ms",        "NT={label};AC={obo_id}"),
    "Comment[AcquisitionMethod]":        ("ms",        "NT={label};AC={obo_id}"),
    "Comment[EnrichmentMethod]":         ("ms",        "NT={label};AC={obo_id}"),
    "Comment[FractionationMethod]":      ("ms",        "NT={label};AC={obo_id}"),
    "Comment[Separation]":               ("ms",        "NT={label};AC={obo_id}"),
}


def ols_lookup(term: str, ontology: str) -> Optional[Tuple[str, str]]:
    """
    Query OLS4 for `term` in `ontology`.
    Returns (label, obo_id) of the best hit, or None.
    """
    key = (term.lower().strip(), ontology)
    if key in _ols_cache:
        return _ols_cache[key]

    try:
        resp = requests.get(
            OLS_BASE,
            params={"q": term, "ontology": ontology, "rows": 5, "exact": "false"},
            timeout=10,
        )
        resp.raise_for_status()
        docs = resp.json().get("response", {}).get("docs", [])
        if docs:
            best = docs[0]
            label  = best.get("label", term)
            obo_id = best.get("obo_id", "")
            result = (label, obo_id)
        else:
            result = None
    except Exception:
        result = None

    _ols_cache[key] = result
    return result


def resolve_to_sdrf_term(raw_value: str, col: str) -> str:
    """
    Given a raw extracted value and the SDRF column name, return the
    ontology-formatted string (NT=...;AC=...) if a mapping exists,
    else return the raw value cleaned up.
    """
    v = raw_value.strip()
    if not v or v.lower() in ("Not Applicable", "not applicable", "na", "n/a", ""):
        return "Not Applicable"

    # Already formatted
    if v.startswith("NT="):
        return v

    if col in COLUMN_ONTOLOGY:
        ontology, fmt = COLUMN_ONTOLOGY[col]
        hit = ols_lookup(v, ontology)
        if hit:
            label, obo_id = hit
            obo_id_clean = obo_id.replace(":", "_") if obo_id else ""
            # Use the AC in colon format for SDRF (e.g. MS:1001251)
            ac = obo_id if ":" in obo_id else obo_id_clean
            return fmt.format(label=label, obo_id=ac)

    return v


# Test the resolver
test_cases = [
    ("Homo sapiens",    "Characteristics[Organism]"),
    ("liver cancer",    "Characteristics[Disease]"),
    ("blood plasma",    "Characteristics[OrganismPart]"),
    ("trypsin",         "Characteristics[CleavageAgent]"),
    ("Q Exactive HF",   "Comment[Instrument]"),
    ("HCD",             "Comment[FragmentationMethod]"),
    ("label free",      "Characteristics[Label]"),
    ("Oxidation",       "Characteristics[Modification]"),
]
for raw, col in test_cases:
    result = resolve_to_sdrf_term(raw, col)
    print(f"  {raw:25s} -> {result}")


  Homo sapiens              -> NCBITaxon:9606 (Homo sapiens)
  liver cancer              -> liver cancer
  blood plasma              -> NT=blood plasma;AC=UBERON:0001969
  trypsin                   -> NT=Trypsin;AC=MS:1001251
  Q Exactive HF             -> AC=MS:1002523;NT=Q Exactive HF
  HCD                       -> AC=MS:1000422;NT=beam-type collision-induced dissociation
  label free                -> label free
  Oxidation                 -> Oxidation


## [FUNC] PubMedBERT NER + extraction

In [6]:
import torch
from transformers import pipeline as hf_pipeline

_ner_pipe = None
NER_CONF   = 0.88
NER_MINLEN = 4

def get_ner():
    global _ner_pipe
    if _ner_pipe is None:
        dev = 0 if torch.cuda.is_available() else -1
        gpu = torch.cuda.get_device_name(0) if dev==0 else 'CPU'
        print(f'Loading {NER_MODEL} on {gpu}...')
        t0 = time.time()
        _ner_pipe = hf_pipeline('ner', model=NER_MODEL,
                                aggregation_strategy='simple', device=dev)
        print(f'NER ready in {time.time()-t0:.1f}s')
    return _ner_pipe

# Tissue keyword → UBERON canonical
TISSUE_MAP = {
    'blood serum':'NT=blood serum;AC=UBERON:0001977',
    'serum':'NT=blood serum;AC=UBERON:0001977',
    'blood plasma':'NT=blood plasma;AC=UBERON:0001969',
    'plasma':'NT=blood plasma;AC=UBERON:0001969',
    'blood':'NT=blood;AC=UBERON:0000178',
    'urine':'NT=urine;AC=UBERON:0001088',
    'cerebrospinal fluid':'NT=cerebrospinal fluid;AC=UBERON:0001359',
    'csf':'NT=cerebrospinal fluid;AC=UBERON:0001359',
    'saliva':'NT=saliva;AC=UBERON:0001836',
    'brain':'NT=brain;AC=UBERON:0000955',
    'frontal cortex':'NT=frontal cortex;AC=UBERON:0001870',
    'prefrontal cortex':'NT=prefrontal cortex;AC=UBERON:0000451',
    'dorsolateral prefrontal cortex':'NT=prefrontal cortex;AC=UBERON:0000451',
    'cerebral cortex':'NT=cerebral cortex;AC=UBERON:0000956',
    'hippocampus':'NT=hippocampal formation;AC=UBERON:0002421',
    'liver':'NT=liver;AC=UBERON:0002107',
    'lung':'NT=lung;AC=UBERON:0002048',
    'heart':'NT=heart;AC=UBERON:0000948',
    'kidney':'NT=kidney;AC=UBERON:0002113',
    'bone marrow':'NT=bone marrow;AC=UBERON:0002371',
    'skin':'NT=skin of body;AC=UBERON:0002097',
    'breast':'NT=breast;AC=UBERON:0000310',
    'prostate gland':'NT=prostate gland;AC=UBERON:0002367',
    'prostate':'NT=prostate gland;AC=UBERON:0002367',
    'adipose tissue':'NT=adipose tissue;AC=UBERON:0001013',
    'adipose':'NT=adipose tissue;AC=UBERON:0001013',
    'colon':'NT=colon;AC=UBERON:0001155',
    'ovary':'NT=ovary;AC=UBERON:0000992',
    'testis':'NT=testis;AC=UBERON:0000473',
    'spleen':'NT=spleen;AC=UBERON:0002106',
    'pancreas':'NT=pancreas;AC=UBERON:0001264',
    'extracellular vesicle':'NT=extracellular vesicle;AC=GO:0061695',
    'exosome':'NT=extracellular vesicle;AC=GO:0061695',
    'pbmc':'NT=peripheral blood mononuclear cell;AC=CL:0000057',
}

KNOWN_CL = {
    'hela':'HeLa','hek293t':'HEK293T','hek293':'HEK293','u2os':'U2OS',
    'mcf7':'MCF7','a549':'A549','jurkat':'Jurkat','k562':'K562',
    'hct116':'HCT116','hepg2':'HepG2','pc3':'PC3','lncap':'LNCaP',
    'thp-1':'THP-1','sh-sy5y':'SH-SY5Y','mda-mb-231':'MDA-MB-231',
    'u937':'U937','huvec':'HUVEC','nih3t3':'NIH3T3','cho':'CHO',
    '3t3-l1':'3T3-L1','raw264.7':'RAW264.7','c2c12':'C2C12',
}

ORGANISM_MAP = {
    'homo sapiens':'9606 (Homo sapiens)','human':'9606 (Homo sapiens)',
    'mus musculus':'10090 (Mus musculus)','mouse':'10090 (Mus musculus)',
    'mice':'10090 (Mus musculus)','murine':'10090 (Mus musculus)',
    'rattus norvegicus':'10116 (Rattus norvegicus)','rat':'10116 (Rattus norvegicus)',
    'escherichia coli':'562 (Escherichia coli)','e. coli':'562 (Escherichia coli)',
    'saccharomyces cerevisiae':'4932 (Saccharomyces cerevisiae)',
    'yeast':'4932 (Saccharomyces cerevisiae)',
    'bos taurus':'9913 (Bos taurus)','bovine':'9913 (Bos taurus)',
    'danio rerio':'7955 (Danio rerio)','zebrafish':'7955 (Danio rerio)',
    'drosophila melanogaster':'7227 (Drosophila melanogaster)',
    'arabidopsis thaliana':'3702 (Arabidopsis thaliana)',
    'sus scrofa':'9823 (Sus scrofa)','pig':'9823 (Sus scrofa)',
}

DISEASE_NORM = {
    "alzheimer's disease":'Alzheimer disease',
    'alzheimer disease':'Alzheimer disease',
    "parkinson's disease":'Parkinson disease',
    'glioblastoma multiforme':'glioblastoma',
    'brain glioblastoma multiforme':'glioblastoma',
    'lung cancer':'lung carcinoma','lung cancer':'lung carcinoma',
    'prostate cancer':'prostate carcinoma',
    'breast cancer':'breast carcinoma',
    'colorectal cancer':'colorectal carcinoma',
    'colon cancer':'colorectal carcinoma',
}

BIOFLUID_TERMS = {'serum','plasma','urine','csf','saliva','sweat','bronchoalveolar'}

_SPIKE_PAT = re.compile(
    r'\b(spike[\s\-]?in|spiked?|standard|control|qc|reference|calibrat)\b', re.I)

def _is_spikein(word, text, window=120):
    for m in re.finditer(re.escape(word.lower()), text.lower()):
        if _SPIKE_PAT.search(text[max(0,m.start()-window):m.end()+window]):
            return True
    return False


def ner_extract(abstract: str) -> Dict[str, str]:
    """
    Run PubMedBERT NER on abstract text.
    Returns flat dict of col → value.
    Uses OLS to resolve extracted values to canonical format.
    """
    out: Dict[str,str] = {}
    if not abstract.strip():
        return out

    pipe = get_ner()
    abs_low = abstract.lower()
    is_biofluid = any(t in abs_low for t in BIOFLUID_TERMS)

    # ── Run NER ─────────────────────────────────────────────────────────────
    try:
        entities = pipe(abstract[:2000])
    except Exception as e:
        print(f'NER error: {e}'); return out

    for ent in entities:
        label = ent.get('entity_group','').upper()
        word  = re.sub(r'\s*##','',ent.get('word','').strip())  # subword stitching
        score = ent.get('score',0)

        if score < NER_CONF or len(word) < NER_MINLEN: continue
        if word.lower() in ('disease','cancer','tumor','normal','healthy',
                            'protein','cell','cells','mice','rats','human'): continue

        wkey = word.lower().replace(' ','').replace('-','')

        # Disease — plain text, normalised
        if label in ('DISEASE','DISEASE_DISORDER'):
            norm = DISEASE_NORM.get(word.lower(), word)
            if 'Characteristics[Disease]' not in out:
                out['Characteristics[Disease]'] = norm

        # Cell line — plain text, not in biofluid study, not a spike-in
        elif label == 'CELL_LINE' and not is_biofluid:
            canonical = KNOWN_CL.get(wkey)
            if canonical and not _is_spikein(word, abstract):
                if 'Characteristics[CellLine]' not in out:
                    out['Characteristics[CellLine]'] = canonical

        # Cell type
        elif label == 'CELL_TYPE':
            if 'Characteristics[CellType]' not in out:
                out['Characteristics[CellType]'] = word

        # Organism
        elif label in ('SPECIES','ORGANISM','TAXON'):
            norm = next((v for k,v in ORGANISM_MAP.items() if k in word.lower()), None)
            if norm and 'Characteristics[Organism]' not in out:
                out['Characteristics[Organism]'] = norm

    # ── Tissue from keyword matching (more reliable than NER for anatomy) ───
    if 'Characteristics[OrganismPart]' not in out:
        for term in sorted(TISSUE_MAP, key=len, reverse=True):
            if re.search(r'\b' + re.escape(term) + r'\b', abs_low):
                tissue_val = TISSUE_MAP[term]
                is_fl = any(t in tissue_val.lower() for t in ['plasma','serum','urine','csf'])
                if is_biofluid == is_fl or not is_biofluid:
                    out['Characteristics[OrganismPart]'] = tissue_val
                    break

    # ── MaterialType inferred ────────────────────────────────────────────────
    if 'Characteristics[CellLine]' in out:
        out['Characteristics[MaterialType]'] = 'cell line'
    elif is_biofluid:
        out['Characteristics[MaterialType]'] = 'biofluid'
    elif 'Characteristics[OrganismPart]' in out:
        out['Characteristics[MaterialType]'] = 'tissue'

    # ── OLS resolution for every value we found ──────────────────────────────
    # OrganismPart already in NT=;AC= format from TISSUE_MAP
    # Disease stays as plain text — do NOT resolve through OLS
    # CellLine stays as plain text
    # Organism already in taxon format
    # CellType — resolve through OLS 'cl' ontology
    if 'Characteristics[CellType]' in out:
        resolved = resolve_to_sdrf_term(out['Characteristics[CellType]'], 'Characteristics[CellType]')
        out['Characteristics[CellType]'] = resolved

    return out

print('PubMedBERT NER functions defined.')
print('NER model will load on first call.')

PubMedBERT NER functions defined.
NER model will load on first call.


## [FUNC] Protocol regex (instrument, enzyme, mods)

In [7]:
# ── Fast local lookup dicts (avoid OLS API for common protocol terms) ────────
INSTRUMENT_ONT = {
    'q exactive hf-x':'AC=MS:1003027;NT=Q Exactive HF-X',
    'q exactive hf':'AC=MS:1002523;NT=Q Exactive HF',
    'q exactive plus':'AC=MS:1002634;NT=Q Exactive Plus',
    'q exactive':'AC=MS:1001911;NT=Q Exactive',
    'orbitrap astral':'AC=MS:1003378;NT=Orbitrap Astral',
    'orbitrap fusion lumos':'AC=MS:1002732;NT=Orbitrap Fusion Lumos',
    'orbitrap fusion':'AC=MS:1002416;NT=Orbitrap Fusion',
    'orbitrap eclipse':'AC=MS:1003029;NT=Orbitrap Eclipse',
    'orbitrap exploris 480':'AC=MS:1003094;NT=Orbitrap Exploris 480',
    'exploris 480':'AC=MS:1003094;NT=Orbitrap Exploris 480',
    'ltq orbitrap velos':'AC=MS:1001742;NT=LTQ Orbitrap Velos',
    'ltq orbitrap elite':'AC=MS:1001910;NT=LTQ Orbitrap Elite',
    'ltq orbitrap xl':'AC=MS:1000556;NT=LTQ Orbitrap XL',
    'ltq orbitrap':'AC=MS:1000449;NT=LTQ Orbitrap',
    'timstof pro 2':'AC=MS:1003474;NT=timsTOF Pro 2',
    'timstof pro':'AC=MS:1003231;NT=timsTOF Pro',
    'timstof':'AC=MS:1002817;NT=timsTOF',
    'triple tof 6600':'AC=MS:1000931;NT=TripleTOF 6600',
    'triple tof 5600':'AC=MS:1000931;NT=TripleTOF 5600',
}
CLEAVAGE_ONT = {
    'trypsin':'AC=MS:1001251;NT=Trypsin',
    'lys-c':'AC=MS:1001255;NT=Lys-C','lysc':'AC=MS:1001255;NT=Lys-C',
    'glu-c':'AC=MS:1001917;NT=Glu-C',
    'chymotrypsin':'AC=MS:1001306;NT=Chymotrypsin',
    'asp-n':'AC=MS:1001267;NT=Asp-N',
}
LABEL_ONT = {
    'label free':'AC=MS:1002038;NT=label free sample',
    'label-free':'AC=MS:1002038;NT=label free sample',
    'lfq':'AC=MS:1002038;NT=label free sample',
    'silac':'AC=MS:1002791;NT=SILAC',
    'dimethyl':'AC=MS:1002457;NT=Dimethyl',
}

_NEG = r'(?<!without\s)(?<!no\s)(?<!not\s)'

def protocol_regex(full_text: str) -> Dict[str,str]:
    """Extract protocol fields. Returns flat dict col→value."""
    out: Dict[str,str] = {}
    t = full_text; tl = t.lower()

    # Instrument — local dict first, OLS fallback
    for key in sorted(INSTRUMENT_ONT, key=len, reverse=True):
        if key in tl:
            out['Comment[Instrument]'] = INSTRUMENT_ONT[key]; break
    if 'Comment[Instrument]' not in out:
        for pat,val in [
            (r'\b(Q[\s\-]?Exactive[\s\-]?HF[\s\-]?X)\b','AC=MS:1003027;NT=Q Exactive HF-X'),
            (r'\b(Q[\s\-]?Exactive[\s\-]?HF)\b','AC=MS:1002523;NT=Q Exactive HF'),
        ]:
            if re.search(pat,t,re.I): out['Comment[Instrument]']=val; break

    # Cleavage agent
    for key in sorted(CLEAVAGE_ONT, key=len, reverse=True):
        if re.search(_NEG+r'\b'+re.escape(key)+r'\b', tl):
            out['Characteristics[CleavageAgent]'] = CLEAVAGE_ONT[key]; break

    # Label
    for pat,fn in [
        (r'\b(tmt[\s\-]?(?:pro|18|16|11|10|6|2)?(?:plex)?)\b', lambda m: (
            lambda n: {'2':'AC=MS:1002456;NT=TMT2plex','6':'AC=MS:1002453;NT=TMT6plex',
                       '10':'AC=MS:1002454;NT=TMT10plex','11':'AC=MS:1002454;NT=TMT11plex',
                       '16':'AC=MS:1003998;NT=TMT16plex','18':'AC=MS:1003999;NT=TMT18plex',
                       'pro':'AC=MS:1003998;NT=TMT16plex'}.get(n,'AC=MS:1002453;NT=TMT6plex')
        )(re.search(r'(pro|18|16|11|10|6|2)',m.group(1),re.I).group(1).lower()
          if re.search(r'(pro|18|16|11|10|6|2)',m.group(1),re.I) else '6')),
        (r'\b(itraq[\s\-]?(?:4|8)?(?:plex)?)\b', lambda m:
            'AC=MS:1001985;NT=iTRAQ4plex' if '4' in m.group(1) else 'AC=MS:1002519;NT=iTRAQ8plex'),
        (r'\b(silac)\b', lambda m: 'AC=MS:1002791;NT=SILAC'),
        (r'\b(label[\s\-]free|lfq)\b', lambda m: 'AC=MS:1002038;NT=label free sample'),
        (r'\b(dimethyl)\b', lambda m: 'AC=MS:1002457;NT=Dimethyl'),
    ]:
        m = re.search(pat, t, re.I)
        if m: out['Characteristics[Label]'] = fn(m); break

    # Reduction
    for pat,val in [
        (_NEG+r'\b(dtt|dithiothreitol)\b','AC=MS:1000578;NT=DTT'),
        (_NEG+r'\b(tcep)\b','AC=MS:1001135;NT=TCEP'),
    ]:
        if re.search(pat,t,re.I): out['Characteristics[ReductionReagent]']=val; break

    # Alkylation
    for pat,val in [
        (r'\b(iodoacetamide|iaa)\b','AC=PRIDE:0000126;NT=Iodoacetamide'),
        (r'\b(chloroacetamide|caa)\b','AC=PRIDE:0000126;NT=Chloroacetamide'),
    ]:
        if re.search(pat,t,re.I): out['Characteristics[AlkylationReagent]']=val; break

    # Modifications
    mods = []
    for pat,val in [
        (r'\b(carbamidomethyl)\b','NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed'),
        (r'\b(oxidation)\b','NT=Oxidation;AC=UNIMOD:35;TA=M;MT=Variable'),
        (r'\b(phospho)\b','NT=Phospho;AC=UNIMOD:21;TA=S,T,Y;MT=Variable'),
        (r'\b(acetyl)\b','NT=Acetyl;AC=UNIMOD:1;TA=K;MT=Variable'),
        (r'\b(deamid)\b','NT=Deamidated;AC=UNIMOD:7;TA=N,Q;MT=Variable'),
    ]:
        if re.search(pat,t,re.I): mods.append(val)
    for i,mod in enumerate(mods[:6]):
        col = 'Characteristics[Modification]' if i==0 else f'Characteristics[Modification].{i}'
        out[col] = mod

    # Acquisition
    for pat,val in [
        (r'\b(dda|data[\s\-]dependent)\b','AC=MS:1003215;NT=DDA'),
        (r'\b(dia|data[\s\-]independent|swath)\b','AC=MS:1003215;NT=DIA'),
    ]:
        if re.search(pat,t,re.I): out['Comment[AcquisitionMethod]']=val; break

    # Fragmentation
    for pat,val in [
        (r'\b(hcd)\b','AC=MS:1002481;NT=HCD'),
        (r'\b(cid)\b','AC=MS:1001880;NT=CID'),
        (r'\b(etd)\b','AC=MS:1001526;NT=ETD'),
    ]:
        if re.search(pat,t,re.I): out['Comment[FragmentationMethod]']=val; break

    # MS2 analyzer
    for pat,val in [
        (r'\b(orbitrap)\b','AC=MS:1000484;NT=Orbitrap'),
        (r'\b(tof)\b','AC=MS:1000084;NT=TOF'),
        (r'\b(ion\s*trap)\b','AC=MS:1000264;NT=ion trap'),
    ]:
        if re.search(pat,t,re.I): out['Comment[MS2MassAnalyzer]']=val; break

    # Ionisation
    for pat,val in [
        (r'\b(nano[\s\-]?esi|nesi)\b','AC=MS:1000398;NT=nanoESI'),
        (r'\b(electrospray|\besi\b)\b','AC=MS:1000073;NT=ESI'),
    ]:
        if re.search(pat,t,re.I): out['Comment[IonizationType]']=val; break

    # Separation
    for pat,val in [
        (r'\b(nano[\s\-]?lc)\b','AC=PRIDE:0000565;NT=nanoLC'),
        (r'\b(rplc|reversed[\s\-]phase)\b','AC=PRIDE:0000550;NT=Reversed-Phase'),
    ]:
        if re.search(pat,t,re.I): out['Comment[Separation]']=val; break

    # Numeric fields
    m = re.search(r'(\d+)[\s\-]min(?:ute)?\s+gradient',t,re.I) or \
        re.search(r'gradient[\s\S]{0,20}?(\d+)[\s\-]?min',t,re.I)
    if m: out['Comment[GradientTime]'] = f'{m.group(1)} min'

    m = re.search(r'(\d+(?:\.\d+)?)\s*(nl|nL)/\s*min',t)
    if m: out['Comment[FlowRateChromatogram]'] = f'{m.group(1)} nL/min'

    m = re.search(r'(\d+(?:\.\d+)?)\s*ppm[\s\S]{0,30}?precursor',t,re.I) or \
        re.search(r'precursor[\s\S]{0,30}?(\d+(?:\.\d+)?)\s*ppm',t,re.I)
    if m: out['Comment[PrecursorMassTolerance]'] = f'{m.group(1)} ppm'

    m = re.search(r'(\d+(?:\.\d+)?)\s*(da|mda)[\s\S]{0,30}?fragment',t,re.I) or \
        re.search(r'fragment[\s\S]{0,30}?(\d+(?:\.\d+)?)\s*(da|mda)',t,re.I)
    if m: out['Comment[FragmentMassTolerance]'] = f'{m.group(1)} Da'

    m = re.search(r'(\d)\s*missed\s*cleavage',t,re.I)
    if m: out['Comment[NumberOfMissedCleavages]'] = m.group(1)

    # OLS fallback for any protocol values not in local dicts
    for col,raw in list(out.items()):
        if col in COLUMN_ONTOLOGY and not raw.startswith('AC=') and not raw.startswith('NT='):
            resolved = resolve_to_sdrf_term(raw, col)
            if resolved != NOT_APPLICABLE:
                out[col] = resolved

    return out

print('Protocol regex defined.')

Protocol regex defined.


## [FUNC] GPT correction map + TF-IDF retrieval

In [11]:
def build_correction_map(
    train_sdrfs: Dict[str, pd.DataFrame],
    train_gpt:   Dict[str, dict],
    threshold:   float = 0.82,
) -> Dict[str, str]:
    """
    For each column, compare GPT-extracted values to gold-standard values.
    When a GPT value is close to (but not equal to) a gold value, record the mapping.
    Returns: {gpt_value -> gold_value}
    """
    correction: Dict[str, str] = {}

    common_pxds = set(train_sdrfs) & set(train_gpt)
    for pxd in common_pxds:
        gold_df  = train_sdrfs[pxd]
        gpt_data = train_gpt[pxd]

        for col in META_COLS:
            # Gold values for this column
            if col not in gold_df.columns:
                continue
            gold_vals = set(
                str(v).strip() for v in gold_df[col].dropna()
                if str(v).strip() not in ("", "nan", "Not Applicable", "Not Applicable")
            )
            if not gold_vals:
                continue

            # GPT values: may be list or single string
            raw_gpt = gpt_data.get(col, [])
            if isinstance(raw_gpt, str):
                raw_gpt = [raw_gpt]
            elif not isinstance(raw_gpt, list):
                raw_gpt = [str(raw_gpt)]

            gpt_vals = {str(v).strip() for v in raw_gpt if str(v).strip()}

            for gv in gpt_vals:
                if gv in gold_vals:
                    continue  # already correct
                # Find closest gold value
                best_gold, best_sim = None, 0.0
                for gold_v in gold_vals:
                    sim = difflib.SequenceMatcher(None, gv.lower(), gold_v.lower()).ratio()
                    if sim > best_sim:
                        best_sim, best_gold = sim, gold_v
                if best_sim >= threshold and best_gold and gv != best_gold:
                    correction[gv] = best_gold

    print(f"Correction map built: {len(correction)} entries")
    return correction


def apply_correction_map(value: str, cmap: Dict[str, str]) -> str:
    """Look up value in correction map; fall back to case-insensitive match."""
    if value in cmap:
        return cmap[value]
    lower = value.lower()
    for k, v in cmap.items():
        if k.lower() == lower:
            return v
    return value


correction_map = build_correction_map(train_sdrfs, train_gpt)

# Show sample corrections
for k, v in list(correction_map.items())[:15]:
    print(f"  {k!r:50s} -> {v!r}")


Correction map built: 0 entries


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train_pxd_list  = sorted(train_papers.keys())
train_text_list = [train_papers[p]["full_text"][:MAX_PAPER_CHARS] for p in train_pxd_list]

vectorizer   = TfidfVectorizer(max_features=15_000, sublinear_tf=True, stop_words="english")
train_matrix = vectorizer.fit_transform(train_text_list)

def retrieve_similar(query_text: str, top_k: int = 3) -> List[Tuple[str, float]]:
    """Return [(pxd, similarity_score), ...] for the top-k closest training papers."""
    q_vec = vectorizer.transform([query_text[:MAX_PAPER_CHARS]])
    sims  = cosine_similarity(q_vec, train_matrix)[0]
    top_i = np.argsort(sims)[::-1][:top_k]
    return [(train_pxd_list[i], float(sims[i])) for i in top_i]

# Quick sanity check
example_pxd  = list(test_papers.keys())[0]
example_hits = retrieve_similar(test_papers[example_pxd]["full_text"])
print(f"Test PXD {example_pxd} → closest training papers:")
for pxd, sim in example_hits:
    print(f"  {pxd}  (sim={sim:.3f})")


Test PXD PubText → closest training papers:
  PubText  (sim=0.000)
  PXD021874  (sim=0.000)
  PXD003531  (sim=0.000)


## [CORE] Main extraction loop

In [12]:
all_pred_rows: List[dict] = []
extraction_log: List[dict] = []

# ── Checkpoint: resume from previous run ──────────────────────────────────────
_ckpt_rows = OUTPUT_DIR / 'checkpoint_rows.json'
_ckpt_log  = OUTPUT_DIR / 'checkpoint_log.json'
_done: set = set()
if _ckpt_rows.exists():
    with open(_ckpt_rows) as f: all_pred_rows = json.load(f)
    _done = {r.get('PXD') for r in all_pred_rows}
    print(f'Resumed: {len(all_pred_rows)} rows, {len(_done)} PXDs done: {_done}')
if _ckpt_log.exists():
    with open(_ckpt_log) as f: extraction_log = json.load(f)

# Load sample submission to get raw file lists per PXD
sample_df = pd.read_csv(SAMPLE_SUB, dtype=str)
pxd_files: Dict[str,List[str]] = {}
for pxd,grp in sample_df.groupby('PXD'):
    pxd_files[pxd] = grp['Raw Data File'].tolist()

print(f'Extracting {len(pxd_files)} test PXDs...\n')

for pxd in tqdm(sorted(pxd_files.keys()), desc='PXDs'):
    if pxd in _done:
        print(f'  Skipping {pxd} (checkpoint)')
        continue

    t0 = time.time()
    paper = test_papers.get(pxd, {})
    full_text = paper.get('full_text', '')
    abstract  = paper.get('ABSTRACT', paper.get('abstract', full_text[:2000]))
    if isinstance(abstract, list): abstract = ' '.join(str(x) for x in abstract)

    # ── Step 1: PubMedBERT NER on abstract ────────────────────────────────────
    ner_vals = ner_extract(abstract) if abstract.strip() else {}

    # ── Step 2: Protocol regex on full text ───────────────────────────────────
    proto_vals = protocol_regex(full_text) if full_text.strip() else {}

    # Merge: NER fills bio cols, regex fills protocol cols
    # NER takes priority for bio cols (PRIDE-like authority)
    merged: Dict[str,str] = {**proto_vals, **ner_vals}

    # ── Step 3: OLS resolve everything not already canonical ──────────────────
    for col, val in list(merged.items()):
        if col in COLUMN_ONTOLOGY and val not in (NOT_APPLICABLE, ''):
            # Disease and CellLine stay as plain text
            if col in ('Characteristics[Disease]','Characteristics[CellLine]',
                       'Characteristics[MaterialType]','Characteristics[Organism]'):
                continue
            if not val.startswith('NT=') and not val.startswith('AC='):
                resolved = resolve_to_sdrf_term(val, col)
                if resolved != NOT_APPLICABLE:
                    merged[col] = resolved

    # ── Step 4: TF-IDF fallback for empty bio cols ─────────────────────────────
    similar = retrieve_similar(full_text or abstract, top_k=3) if (full_text or abstract).strip() else []
    BIO_COLS = {'Characteristics[Organism]','Characteristics[OrganismPart]',
                'Characteristics[Disease]','Characteristics[MaterialType]',
                'Characteristics[CellLine]'}
    for col in BIO_COLS:
        if col in merged: continue
        for sim_pxd, sim_score in similar:
            if sim_score < 0.15: break
            if sim_pxd not in train_sdrfs: continue
            sim_df = train_sdrfs[sim_pxd]
            if col not in sim_df.columns: continue
            vals = sim_df[col].dropna().astype(str)
            vals = vals[~vals.str.lower().isin(['not applicable','nan',''])]
            if not vals.empty:
                top = vals.value_counts().index[0]
                merged[col] = apply_correction_map(top, correction_map)
                break

    # ── Step 5: Build one row per file ────────────────────────────────────────
    raw_files_list = pxd_files[pxd]
    for file_idx, raw_file in enumerate(raw_files_list):
        row = {c: NOT_APPLICABLE for c in ALL_COLS}
        row['PXD']           = pxd
        row['Raw Data File'] = raw_file
        row['Usage']         = 'raw'

        # BiologicalReplicate from filename
        br_m = re.search(r'[_\-](?:br|rep|r)(\d{1,2})[_\-\.]',raw_file,re.I)
        row['Characteristics[BiologicalReplicate]'] = br_m.group(1) if br_m else str(file_idx+1)

        # Fill from merged extraction
        for col, val in merged.items():
            if col in row and val not in ('', None):
                row[col] = apply_correction_map(str(val), correction_map)

        # Fraction identifier from filename
        frac_m = re.search(r'[_\-](f|fr|frac)(\d{1,3})[_\-\.]',raw_file,re.I)
        if frac_m:
            row['Comment[FractionIdentifier]'] = frac_m.group(2)

        all_pred_rows.append(row)

    dt = time.time()-t0
    extraction_log.append({'pxd':pxd,'n_rows':len(raw_files_list),'time_s':round(dt,1),
                            'has_text':bool(full_text),'ner_cols':len(ner_vals),
                            'proto_cols':len(proto_vals)})
    print(f'  {pxd}: {len(raw_files_list)} rows, {len(ner_vals)} NER cols, '
          f'{len(proto_vals)} proto cols, {dt:.1f}s')

    # Checkpoint
    with open(_ckpt_rows,'w') as f: json.dump(all_pred_rows, f)
    with open(_ckpt_log,'w') as f: json.dump(extraction_log, f)

print(f'\nDone. {len(all_pred_rows)} total rows.')
pd.DataFrame(extraction_log)

Extracting 15 test PXDs...



PXDs:   7%|▋         | 1/15 [00:00<00:10,  1.35it/s]

  PXD004010: 10 rows, 2 NER cols, 2 proto cols, 0.7s
  PXD016436: 18 rows, 0 NER cols, 9 proto cols, 0.2s


PXDs:  20%|██        | 3/15 [00:01<00:03,  3.00it/s]

  PXD019519: 6 rows, 0 NER cols, 9 proto cols, 0.2s
  PXD025663: 12 rows, 3 NER cols, 9 proto cols, 0.2s


...

PXDs:  93%|█████████▎| 14/15 [00:05<00:00,  1.52it/s]

  PXD064564: 30 rows, 0 NER cols, 5 proto cols, 0.2s


PXDs: 100%|██████████| 15/15 [00:06<00:00,  2.19it/s]


Done. 1659 total rows.


,pxd,n_rows,time_s,has_text,ner_cols,proto_cols
0,PXD004010,10,0.7,True,2,2
1,PXD016436,18,0.2,True,0,9
2,PXD019519,6,0.2,True,0,9
3,PXD025663,12,0.2,True,3,9
4,PXD040582,24,0.1,True,1,11
5,PXD050621,9,0.1,True,0,1
6,PXD061009,2,0.2,True,3,10
7,PXD061090,6,0.2,True,0,1
8,PXD061136,2,0.1,True,2,2
9,PXD061195,1376,0.1,True,0,13


## [EXPORT] Build and validate submission

In [17]:
submission_df = pd.DataFrame(all_pred_rows)

# Assign IDs positionally: both the sample submission and our rows are in
# sorted-PXD order (sample sub is sorted by PXD; extraction loop used sorted()).
# A merge on (PXD, Raw Data File) would explode because PXD061195 reuses the
# same raw file name across multiple TMT-channel rows.
_id_ref = pd.read_csv(SAMPLE_SUB, usecols=['ID','PXD']).sort_values('PXD', kind='stable')
assert len(submission_df) == len(_id_ref), \
    f"Row count mismatch: generated {len(submission_df)} vs sample {len(_id_ref)}"
submission_df['ID'] = _id_ref['ID'].astype(int).values

# Ensure all columns present and ordered correctly
for col in ALL_COLS:
    if col not in submission_df.columns:
        submission_df[col] = NOT_APPLICABLE
submission_df = submission_df[ALL_COLS]

# Fix semicolon spacing (AC=MS:1000484; NT= → AC=MS:1000484;NT=)
for col in META_COLS:
    submission_df[col] = submission_df[col].astype(str).str.replace(r';\s+',';',regex=True)

# Spot checks
print(f'Shape: {submission_df.shape}')
print(f'ID dupes: {submission_df["ID"].duplicated().sum()}  (must be 0)')
for col in ['Characteristics[SyntheticPeptide]','Characteristics[Bait]']:
    if col in submission_df:
        n = (submission_df[col]==NOT_APPLICABLE).sum()
        print(f'  {col}: {n}/{len(submission_df)} NA {"✓" if n==len(submission_df) else "✗"}')

# Fill rate
rows = [(c,(submission_df[c]!=NOT_APPLICABLE).sum()) for c in META_COLS]
rows.sort(key=lambda x:-x[1])
print(f'\nFill rate (top 20):')
for col,n in rows[:20]:
    if n>0: print(f'  {col:<52} {n:>5} {n/len(submission_df)*100:>5.1f}%')
print(f'\nTotal filled: {sum(1 for _,n in rows if n>0)}/{len(rows)}')

# PXD summary
print('\nPXD summary:')
for pxd,grp in submission_df.groupby('PXD'):
    org  = grp['Characteristics[Organism]'].value_counts().index[0]
    part = grp['Characteristics[OrganismPart]'].value_counts().index[0]
    dis  = grp['Characteristics[Disease]'].value_counts().index[0]
    print(f'  {pxd} ({len(grp):4d}): org={str(org)[:20]:<20} '
          f'part={str(part)[:28]:<28} dis={str(dis)[:18]}')


Shape: (1659, 81)
ID dupes: 0  (must be 0)
  Characteristics[SyntheticPeptide]: 1659/1659 NA ✓
  Characteristics[Bait]: 1659/1659 NA ✓

Fill rate (top 20):
  Characteristics[BiologicalReplicate]                  1659 100.0%
  Characteristics[CleavageAgent]                        1610  97.0%
  Characteristics[ReductionReagent]                     1602  96.6%
  Comment[AcquisitionMethod]                            1554  93.7%
  Comment[FragmentationMethod]                          1554  93.7%
  Characteristics[Modification]                         1534  92.5%
  Characteristics[AlkylationReagent]                    1516  91.4%
  Characteristics[Label]                                1479  89.2%
  Comment[FlowRateChromatogram]                         1460  88.0%
  Comment[IonizationType]                               1456  87.8%
  Comment[GradientTime]                                 1454  87.6%
  Comment[Separation]                                   1444  87.0%
  Comment[FragmentMassTolera

## [EXPORT] Save

In [18]:
sub_path = OUTPUT_DIR / 'submission.csv'
submission_df.to_csv(sub_path, index=False)
print(f'Saved: {sub_path}  shape={submission_df.shape}')

# Save OLS cache for next run
ols_pkl = OUTPUT_DIR / 'ols_cache.pkl'
with open(ols_pkl,'wb') as f: pickle.dump(_ols_cache, f)
print(f'OLS cache saved: {ols_pkl} ({len(_ols_cache)} entries)')

# Verify column structure
missing = set(ALL_COLS) - set(submission_df.columns)
if missing: print(f'MISSING cols: {missing}')
else: print('Column structure: ✓')

Saved: C:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\outputs\submission.csv  shape=(1659, 81)
OLS cache saved: C:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\outputs\ols_cache.pkl (8 entries)
Column structure: ✓
